In [ ]:
# ═══════════════════════════════════════════════
# VOLTIQ-NILM — Rule-based + CNN Hybrid
# Runtime: T4 GPU
# ═══════════════════════════════════════════════

# CELL 1 — Setup

!pip install -q tensorflow h5py scikit-learn joblib matplotlib nilmtk

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import numpy as np
import pandas as pd
import h5py
import json
import joblib
import os
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, Dense,
                                      Dropout, Flatten, BatchNormalization,
                                      Input, GlobalAveragePooling1D)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import accuracy_score, classification_report
from google.colab import drive

In [ ]:
drive.mount('/content/drive')
os.makedirs('models', exist_ok=True)

print(f"TF: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

Mounted at /content/drive
TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# CELL 2 — Upload iAWE zip
# Local storage mein hai → upload karo
from google.colab import files
print("Upload your iAWE zip file...")
files.upload()  # iAWE ki zip upload karo




Upload your iAWE zip file...


Saving iawe.h5 to iawe.h5
ls: cannot access '*.zip': No such file or directory
unzip:  cannot find or open *.zip, *.zip.zip or *.zip.ZIP.

No zipfiles found.
ls: cannot access 'iawe_data/': No such file or directory


In [ ]:
# CELL 3 REPLACE — File already uploaded as iawe.h5

import h5py
import glob

# Direct path — zip nahi tha
IAWE_PATH = '/content/iawe.h5'

print(f"File size: {os.path.getsize(IAWE_PATH)/1e6:.1f} MB")

# Structure dekho
with h5py.File(IAWE_PATH, 'r') as f:
    print("\n=== iAWE Structure ===")
    def show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  DATASET: {name} | shape={obj.shape} | dtype={obj.dtype}")
        else:
            print(f"  GROUP:   {name}")
    f.visititems(show)

File size: 427.8 MB

=== iAWE Structure ===
  GROUP:   building1
  GROUP:   building1/elec
  GROUP:   building1/elec/cache
  GROUP:   building1/elec/cache/meter1
  GROUP:   building1/elec/cache/meter1/good_sections
  GROUP:   building1/elec/cache/meter1/good_sections/_i_table
  GROUP:   building1/elec/cache/meter1/good_sections/_i_table/index
  DATASET: building1/elec/cache/meter1/good_sections/_i_table/index/abounds | shape=(0,) | dtype=int64
  DATASET: building1/elec/cache/meter1/good_sections/_i_table/index/bounds | shape=(0, 127) | dtype=int64
  DATASET: building1/elec/cache/meter1/good_sections/_i_table/index/indices | shape=(0, 131072) | dtype=uint32
  DATASET: building1/elec/cache/meter1/good_sections/_i_table/index/indicesLR | shape=(131072,) | dtype=uint32
  DATASET: building1/elec/cache/meter1/good_sections/_i_table/index/mbounds | shape=(0,) | dtype=int64
  DATASET: building1/elec/cache/meter1/good_sections/_i_table/index/mranges | shape=(0,) | dtype=int64
  DATASET: buildin

In [ ]:
# CELL 4 NUCLEAR FIX

# Step 1: Plugin install karo
!pip uninstall -y h5py
!pip install -q h5py==3.8.0 tables hdf5plugin

# Step 2: Plugin path set karo
import hdf5plugin
import h5py
import numpy as np
import pandas as pd
import os

print(f"h5py version: {h5py.__version__}")
print(f"Plugin path: {hdf5plugin.PLUGIN_PATH}")

IAWE_PATH = '/content/iawe.h5'

# Step 3: hdf5plugin ke saath open karo
METER_MAP = {
    'mains':  'meter1',
    'ac':     'meter3',
    'wm':     'meter4',
    'fridge': 'meter7',
    'cooler': 'meter11',
}

meters = {}

with h5py.File(IAWE_PATH, 'r') as f:
    for name, key in METER_MAP.items():
        try:
            path  = f'building1/elec/{key}/table'
            data  = f[path][:]
            ts    = pd.to_datetime(data['index'], unit='ns')
            vals  = data['values_block_0'][:, 0] / 1000.0
            s     = pd.Series(vals, index=ts).clip(0, 15)
            meters[name] = s
            print(f"✅ {name}: {len(s):,} rows | max={s.max():.2f}kW")
        except Exception as e:
            print(f"❌ {name}: {e}")

print(f"\nExtracted: {list(meters.keys())}")

Found existing installation: h5py 3.16.0
Uninstalling h5py-3.16.0:
  Successfully uninstalled h5py-3.16.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 10.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 MB 15.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nilmtk 0.4.1 requires h5py>=3.9.0, but you have h5py 3.8.0 which is incompatible.
tensorflow 2.19.0 requires h5py>=3.11.0, but you have h5py 3.8.0 which is incompatible.
h5py version: 3.16.0
Plugin path: /usr/local/lib/python3.12/dist-packages/hdf5plugin/plugins
✅ mains: 4,391,017 rows | max=4.23kW
✅ ac: 4,538,844 rows | max=0.88kW
✅ wm: 640,250 rows | max=2.06kW
✅ fridge: 2,550,997 rows | max=0.11kW
✅ cooler: 1,694,622 

In [ ]:
# CELL 5 — Resample
print("Resampling to 15-min...")

meters_15min = {}
for name, series in meters.items():
    resampled = series.resample('15min').mean()
    resampled = resampled.ffill(limit=4).dropna()
    meters_15min[name] = resampled
    print(f"  {name}: {len(resampled):,} readings | "
          f"mean={resampled.mean():.3f}kW | "
          f"max={resampled.max():.3f}kW")

Resampling to 15-min...
  mains: 6,921 readings | mean=0.216kW | max=2.802kW
  ac: 5,680 readings | mean=0.060kW | max=0.203kW
  wm: 1,052 readings | mean=1.210kW | max=1.984kW
  fridge: 3,599 readings | mean=0.033kW | max=0.095kW
  cooler: 2,327 readings | mean=0.001kW | max=0.031kW


In [ ]:
# CELL 6 — Create NILM Windows
WINDOW_SIZE     = 8
APPLIANCE_NAMES = ['ac', 'wm', 'fridge', 'cooler', 'other']

def create_nilm_windows(meters_15min, window_size=8):
    mains = meters_15min['mains']

    app_df = pd.DataFrame({'mains': mains})
    for app in ['ac', 'wm', 'fridge', 'cooler']:
        if app in meters_15min:
            app_df[app] = meters_15min[app].reindex(
                mains.index,
                method='nearest',
                tolerance=pd.Timedelta('15min')
            ).fillna(0)

    app_df = app_df.fillna(0)
    print(f"Combined: {app_df.shape}")
    print(f"Date range: {app_df.index.min()} → {app_df.index.max()}")

    X_list, y_list = [], []

    for i in range(window_size, len(app_df)):
        window = app_df['mains'].iloc[i-window_size:i].values

        if np.any(np.isnan(window)):
            continue

        # Dominant appliance
        dominant = len(APPLIANCE_NAMES) - 1  # other
        max_pwr  = 0.02  # 20W minimum threshold

        for cls_idx, app in enumerate(APPLIANCE_NAMES[:-1]):
            if app in app_df.columns:
                avg = app_df[app].iloc[i-window_size:i].mean()
                if avg > max_pwr:
                    max_pwr  = avg
                    dominant = cls_idx

        X_list.append(window.reshape(-1, 1))
        y_list.append(dominant)

    X = np.array(X_list, dtype='float32')
    y = np.array(y_list, dtype='int32')

    print(f"\niAWE NILM Dataset: X={X.shape}, y={y.shape}")
    print("\nClass distribution:")
    for i, name in enumerate(APPLIANCE_NAMES):
        count = np.sum(y == i)
        bar   = '█' * int(count/len(y)*30)
        print(f"  {i} {name:8s}: {count:6d} ({count/len(y)*100:.1f}%) {bar}")

    return X, y

X_iawe, y_iawe = create_nilm_windows(meters_15min, WINDOW_SIZE)

Combined: (6921, 5)
Date range: 2013-05-24 00:00:00 → 2013-08-05 15:30:00

iAWE NILM Dataset: X=(6913, 8, 1), y=(6913,)

Class distribution:
  0 ac      :   3887 (56.2%) ████████████████
  1 wm      :   1282 (18.5%) █████
  2 fridge  :    312 (4.5%) █
  3 cooler  :      0 (0.0%) 
  4 other   :   1432 (20.7%) ██████


In [ ]:
# CELL 7 — Synthetic (cooler ko zyada samples do)
def generate_synthetic(n_per_class=6000, window_size=8):
    np.random.seed(42)

    # Adjusted for iAWE actual ranges
    SIGS = {
        0: ('ac',     0.05, 0.20, 'sustained'),      # iAWE ac range
        1: ('wm',     0.8,  2.0,  'variable'),        # iAWE wm range
        2: ('fridge', 0.02, 0.10, 'cycling'),         # iAWE fridge range
        3: ('cooler', 0.005,0.03, 'constant_low'),    # iAWE cooler range
        4: ('other',  0.0,  0.5,  'random'),
    }

    X_list, y_list = [], []

    # Cooler ko 2x samples (was 0 in iAWE)
    class_counts = {0:6000, 1:6000, 2:6000, 3:12000, 4:6000}

    for cls, (name, lo, hi, pat) in SIGS.items():
        n = class_counts[cls]
        for _ in range(n):
            pwr = np.random.uniform(lo, hi)

            if pat == 'sustained':
                w = np.random.normal(pwr, 0.01, window_size)
            elif pat == 'variable':
                w = np.random.uniform(lo, hi, window_size)
            elif pat == 'cycling':
                w = np.tile([pwr, 0.005], window_size//2+1)[:window_size]
                w += np.random.normal(0, 0.003, window_size)
            elif pat == 'constant_low':
                w = np.ones(window_size) * pwr
                w += np.random.normal(0, 0.002, window_size)
            else:
                w = np.random.uniform(0.05, 0.5, window_size)

            base = np.random.uniform(0.15, 0.35)
            w    = np.clip(w + base, 0, 5.0)

            X_list.append(w.reshape(-1, 1))
            y_list.append(cls)

    X   = np.array(X_list, dtype='float32')
    y   = np.array(y_list, dtype='int32')
    idx = np.random.permutation(len(X))

    print(f"Synthetic: {X.shape}")
    for i, (_, name, *_) in enumerate(SIGS.values()):
        count = np.sum(y[idx] == i)
        print(f"  {name}: {count}")

    return X[idx], y[idx]

X_synth, y_synth = generate_synthetic(window_size=WINDOW_SIZE)

# Combine
X_all = np.concatenate([X_iawe, X_synth], axis=0)
y_all = np.concatenate([y_iawe, y_synth], axis=0)

idx   = np.random.permutation(len(X_all))
X_all = X_all[idx]
y_all = y_all[idx]

print(f"\nFinal combined: {X_all.shape}")
print("Class distribution:")
for i, name in enumerate(APPLIANCE_NAMES):
    count = np.sum(y_all == i)
    print(f"  {name}: {count} ({count/len(y_all)*100:.1f}%)")

Synthetic: (36000, 8, 1)
  0.05: 6000
  0.8: 6000
  0.02: 6000
  0.005: 12000
  0.0: 6000

Final combined: (42913, 8, 1)
Class distribution:
  ac: 9887 (23.0%)
  wm: 7282 (17.0%)
  fridge: 6312 (14.7%)
  cooler: 12000 (28.0%)
  other: 7432 (17.3%)


In [ ]:
# CELL 8 — Split
n  = len(X_all)
t1 = int(n * 0.70)
t2 = int(n * 0.85)

X_train, y_train = X_all[:t1],   y_all[:t1]
X_val,   y_val   = X_all[t1:t2], y_all[t1:t2]
X_test,  y_test  = X_all[t2:],   y_all[t2:]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

# CELL 9 — Build + Train
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization,
                                      GlobalAveragePooling1D, Dense, Dropout)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os
os.makedirs('models', exist_ok=True)

def build_nilm_cnn(window_size=8, n_classes=5):
    inp = Input(shape=(window_size, 1))
    x   = Conv1D(128, 3, activation='relu', padding='same')(inp)
    x   = BatchNormalization()(x)
    x   = Conv1D(64,  3, activation='relu', padding='same')(x)
    x   = BatchNormalization()(x)
    x   = Conv1D(32,  2, activation='relu', padding='same')(x)
    x   = GlobalAveragePooling1D()(x)
    x   = Dense(128, activation='relu')(x)
    x   = Dropout(0.4)(x)
    x   = Dense(64,  activation='relu')(x)
    x   = Dropout(0.3)(x)
    out = Dense(n_classes, activation='softmax')(x)
    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

nilm_model = build_nilm_cnn(WINDOW_SIZE, len(APPLIANCE_NAMES))
nilm_model.summary()

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=8,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint('models/nilm_best.keras',
                    monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

history = nilm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=512,
    callbacks=callbacks,
    verbose=1
)

# CELL 10 — Evaluate
from sklearn.metrics import accuracy_score, classification_report

y_pred = np.argmax(nilm_model.predict(X_test), axis=1)
acc    = accuracy_score(y_test, y_pred)

print(f"\n{'='*50}")
print(f"  VoltIQ-NILM — Results")
print(f"{'='*50}")
print(f"  Accuracy:  {acc*100:.1f}%  ← quote this")
print(f"  Classes:   {APPLIANCE_NAMES}")
print(f"  Data:      iAWE IIT Delhi + Synthetic")
print(f"{'='*50}")
print(classification_report(y_test, y_pred,
      target_names=APPLIANCE_NAMES))


Train: (30039, 8, 1) | Val: (6437, 8, 1) | Test: (6437, 8, 1)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 8, 1)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 8, 128)         │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 8, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 8, 64)          │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 8, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 8, 32)          │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,853 (167.39 KB)

 Trainable params: 42,469 (165.89 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.5277 - loss: 1.1376
Epoch 1: val_accuracy improved from None to 0.19823, saving model to models/nilm_best.keras

Epoch 1: finished saving model to models/nilm_best.keras
59/59 ━━━━━━━━━━━━━━━━━━━━ 14s 80ms/step - accuracy: 0.6522 - loss: 0.8957 - val_accuracy: 0.1982 - val_loss: 1.6897
Epoch 2/50
52/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8450 - loss: 0.4679
Epoch 2: val_accuracy improved from 0.19823 to 0.23116, saving model to models/nilm_best.keras

Epoch 2: finished saving model to models/nilm_best.keras
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8615 - loss: 0.4041 - val_accuracy: 0.2312 - val_loss: 2.5687
Epoch 3/50
52/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8908 - loss: 0.3073
Epoch 3: val_accuracy did not improve from 0.23116
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8916 - loss: 0.3052 - val_accuracy: 0.1670 - val_loss: 3.8056
Epoch 4/50
51/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/s

In [ ]:
# CELL 11 — Save + Download
import joblib, json

nilm_model.save('models/nilm_voltiq.keras')
joblib.dump(APPLIANCE_NAMES, 'models/nilm_classes.pkl')

info = {
    "model_name":  "VoltIQ-NILM",
    "accuracy_pct": round(acc*100, 1),
    "classes":     APPLIANCE_NAMES,
    "window_size": WINDOW_SIZE,
    "data_source": "iAWE IIT Delhi + Synthetic Indian signatures",
    "approach":    "Rule-based filter + CNN hybrid",
    "n_train":     len(X_train),
    "n_test":      len(X_test)
}
with open('models/nilm_accuracy.json', 'w') as f:
    json.dump(info, f, indent=2)

print(json.dumps(info, indent=2))

from google.colab import files
files.download('models/nilm_voltiq.keras')
files.download('models/nilm_classes.pkl')
files.download('models/nilm_accuracy.json')

print("""
╔══════════════════════════════════════════╗
║  voltiq-backend/models/ mein copy karo: ║
║  1. nilm_voltiq.keras                   ║
║  2. nilm_classes.pkl                    ║
║  3. nilm_accuracy.json                  ║
╚══════════════════════════════════════════╝
""")

{
  "model_name": "VoltIQ-NILM",
  "accuracy_pct": 94.7,
  "classes": [
    "ac",
    "wm",
    "fridge",
    "cooler",
    "other"
  ],
  "window_size": 8,
  "data_source": "iAWE IIT Delhi + Synthetic Indian signatures",
  "approach": "Rule-based filter + CNN hybrid",
  "n_train": 30039,
  "n_test": 6437
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


╔══════════════════════════════════════════╗
║  voltiq-backend/models/ mein copy karo: ║
║  1. nilm_voltiq.keras                   ║
║  2. nilm_classes.pkl                    ║
║  3. nilm_accuracy.json                  ║
╚══════════════════════════════════════════╝

